# Key recovery desde diferenciales ya calculados

Este notebook ejecuta recuperacion de clave para SPN16, AES reducido y KLEIN reducido usando diferenciales conocidos. No vuelve a buscar diferenciales; solo usa `Delta_in` y, para AES/KLEIN, `Delta` en la penultima ronda.

In [1]:
from pathlib import Path
import sys


def add_repo_root_to_path():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "cryptosystems").is_dir() and (path / "differential_cryptoanalysis").is_dir():
            if str(path) not in sys.path:
                sys.path.insert(0, str(path))
            return path
    raise RuntimeError("No se encontro la raiz del repositorio")


ROOT = add_repo_root_to_path()
print(f"Repo root: {ROOT}")

from differential_cryptoanalysis.key_recovery_from_differentials import (
    print_aes_result,
    print_klein_result,
    print_spn16_result,
    recover_reduced_aes_from_differential,
    recover_reduced_klein_from_differential,
    recover_spn16_from_differential,
)

Repo root: C:\Users\juanc\github\Differential-Cryptanalysis-


## SPN16

Aqui si se generan pares elegidos desde textos claros: `P` y `P ^ Delta_in`. El diccionario `expected_delta_u_by_nibble` usa posiciones de nibble `0..3`, donde `0` es el nibble mas significativo.

In [6]:
spn16_result = recover_spn16_from_differential(
    master_key_hex="00112233445566778899AABB",
    rounds=5,
    delta_in=0x0B00,
    expected_delta_u_by_nibble={2: 0x5},
    n_pairs=8000,
    seed=2026,
)

print_spn16_result(spn16_result, top=5)

=== SPN16 key recovery ===
Rondas:                  5
Delta entrada:           0x0B00
Pares elegidos:          8000 (seed=2026)
K_(r+1) real:            0xAABB
K_(r+1) parcial:         0x00B0
Nibbles atacados:        pos 0 = nibble mas significativo
  pos 2: DeltaU=5, rec=B, real=B [OK]
  scores pos 2: B:703, 3:643, 2:592, A:587, D:569


## AES reducido

Para AES reducido se usan pares correctos sinteticos en la entrada de la ultima ronda. Esto permite probar la fase de recuperacion de `K_r` usando el diferencial de la penultima ronda que ya fue calculado.

In [3]:
aes_result = recover_reduced_aes_from_differential(
    master_key_hex="0011223344556677",
    rounds=4,
    block_bits=64,
    key_bits=64,
    delta_in=0x1800000000000000,
    delta_penultimate=0xB8D172C63DC37207,
    n_pairs=64,
    seed=2026,
)

print_aes_result(aes_result, top=5)

=== ReducedAES key recovery ===
Rondas/bloque/clave:     4/64/64
Delta entrada trail:     0x1800000000000000
Delta penultima ronda:   0xB8D172C63DC37207
Pares usados:            64 correctos sinteticos (seed=2026)
K_r real:                0x1110C602F2B48A30
K_r recuperada:          0x1110C602F2B48A30
Coincide:                True
  byte 0: 11:64, A2:4, 06:2, 15:2, 4E:2
  byte 1: 10:64, CB:3, 06:2, 27:2, 3A:2
  byte 2: C6:64, 04:3, 07:2, 11:2, 1F:2
  byte 3: 02:64, 25:3, E0:3, EF:3, 14:2
  byte 4: F2:64, F8:4, 0C:3, FE:3, 09:2
  byte 5: B4:64, 60:3, 19:2, 3B:2, 4B:2
  byte 6: 8A:64, 51:6, 72:5, A9:4, 17:3
  byte 7: 30:64, 73:4, 1E:3, 26:3, 43:3


## KLEIN reducido

Para KLEIN reducido se recuperan nibbles activos de `T = InvRotate(InvMix(K_(r+1)))`. Los nibbles no activos quedan como `?`; si el espacio es pequeno, se enumeran candidatos de `K_(r+1)`.

In [4]:
klein_result = recover_reduced_klein_from_differential(
    master_key_hex="0011223344556677",
    rounds=6,
    block_bits=32,
    key_bits=64,
    delta_in=0x10000000,
    delta_penultimate=0xF07BCBB0,
    n_pairs=128,
    seed=2026,
)

print_klein_result(klein_result, top=5)

=== ReducedKLEIN key recovery ===
Rondas/bloque/clave:     6/32/64
Delta entrada trail:     0x10000000
Delta penultima ronda:   0xF07BCBB0
Pares usados:            128 correctos sinteticos (seed=2026)
K_(r+1) real:            0xAB677121
T real:                  0x3D7D5884
T parcial recuperada:    0x3?7D588?
Candidatos estimados:    256
K real en candidatos:    True
  nibble 0: 3:128, 2:24, E:21, A:17, F:16
  nibble 2: 7:128, 2:48, 9:27, C:27, 8:18
  nibble 3: D:128, F:43, 8:29, A:29, C:21
  nibble 4: 5:128, 3:50, F:48, 9:31, C:18
  nibble 5: 8:128, A:55, D:35, F:35, C:22
  nibble 6: 8:128, A:47, D:32, F:32, 9:19


## Como cambiar el diferencial

- SPN16: cambia `delta_in` y `expected_delta_u_by_nibble`.
- AES/KLEIN: cambia `delta_in` solo para documentar el trail y `delta_penultimate` para la recuperacion real de la ultima ronda.
- Aumenta `n_pairs` si hay ruido o si el ranking de candidatos no separa bien al correcto.